# 31 — Memory Allocator: malloc / free (Amazon FAR style)

You're given a single contiguous region of memory of capacity `c` bytes. Design and
implement `malloc(size)` and `free(addr)` on top of it. As usual the interviewer keeps
piling on constraints — alignment, concurrency, hostile input, then scale.

**Run order:** each part builds on the previous one. Stubs raise `NotImplementedError`;
fill them in and run the matching `--- Part N tests ---` cell. Reference solutions are
collapsed at the very bottom.

## Part 1 — First-fit allocator with coalescing

Implement `Allocator(capacity)`, `malloc(size) -> addr | None`, and `free(addr)`.

**Say this out loud (lock down the spec):**
- Addresses are integer byte offsets in `[0, capacity)`; `malloc` returns the base offset.
- First-fit: scan free blocks in address order, take the first that fits, split the remainder back.
- `free` takes a base address previously returned by `malloc`. On free, **coalesce** with
  adjacent free blocks so fragmentation doesn't accumulate.
- Out of space → return `None` (don't raise). Assume valid `free` calls for now (Part 4 hardens this).

**Things to discuss:**
- Free-list representation: `(offset, size)` blocks sorted by offset vs. a size-indexed structure.
- Why coalescing matters: without it, free/alloc churn permanently fragments memory.
- Complexity: first-fit + coalesce is O(#free blocks) per op; mention segregated free lists for O(1)-ish.

In [ ]:
class Allocator:
    """Fixed-capacity allocator over addresses [0, capacity)."""

    def __init__(self, capacity):
        raise NotImplementedError

    def malloc(self, size):
        """Reserve `size` bytes (first-fit). Return the base address, or None if it doesn't fit."""
        raise NotImplementedError

    def free(self, address):
        """Release a block previously returned by malloc; coalesce adjacent free space."""
        raise NotImplementedError

In [ ]:
# --- Part 1 tests ---
def _t1():
    a = Allocator(100)
    assert a.malloc(10) == 0
    assert a.malloc(20) == 10
    assert a.malloc(70) == 30
    assert a.malloc(1) is None            # full
    a.free(0)                             # frees [0, 10)
    assert a.malloc(8) == 0               # first-fit reuses the front hole
    assert a.malloc(5) is None            # only [8, 10) left, too small
    # coalescing: two frees that touch must merge into one big hole
    b = Allocator(100)
    x, y = b.malloc(50), b.malloc(50)
    assert (x, y) == (0, 50)
    b.free(x); b.free(y)
    assert b.malloc(100) == 0
    print("Part 1 OK")

_t1()

## Part 2 — Alignment, best-fit, and stats

Extend the allocator:
- `malloc(size, align=1)` — the returned address must be a multiple of `align` (a power of two).
  Any padding skipped for alignment goes back to the free list.
- A `policy` of `"best"` picks the **tightest** fitting hole instead of the first.
- `stats()` returns `{capacity, free, used, largest_free, num_free_blocks}`.

**Say this out loud:**
- Alignment is requested per allocation (SIMD / cache line / page boundaries); `align` is a power of two.
- Best-fit reduces external fragmentation on varied sizes but costs a full scan; first-fit is faster.
- `largest_free` is the real "can this big request succeed" number — not total free bytes.

**Things to discuss:** fragmentation metric (`1 - largest_free / free`), best-fit's worst case, when first-fit wins.

In [ ]:
class Allocator:
    """Extend Part 1: power-of-two alignment, best-fit policy, and stats()."""

    def __init__(self, capacity, policy="first"):   # policy: "first" or "best"
        raise NotImplementedError

    def malloc(self, size, align=1):
        raise NotImplementedError

    def free(self, address):
        raise NotImplementedError

    def stats(self):
        """Return {capacity, free, used, largest_free, num_free_blocks}."""
        raise NotImplementedError

In [ ]:
# --- Part 2 tests ---
def _t2():
    # alignment: the address must respect the requested power-of-two boundary
    a = Allocator(100)
    assert a.malloc(1) == 0               # occupies [0, 1)
    addr = a.malloc(8, align=8)
    assert addr == 8 and addr % 8 == 0    # skip to the next multiple of 8
    assert a.malloc(7) == 1               # the [1, 8) alignment gap is reclaimable

    # best-fit picks the tightest hole, unlike first-fit
    b = Allocator(100, policy="best")
    x, y, z, w = b.malloc(40), b.malloc(10), b.malloc(20), b.malloc(10)
    assert (x, y, z, w) == (0, 40, 50, 70)
    b.free(x)                             # hole [0, 40), size 40
    b.free(z)                             # hole [50, 70), size 20
    assert b.malloc(20) == 50             # best-fit -> the size-20 hole (first-fit would give 0)

    # stats accounting always balances
    c = Allocator(64)
    c.malloc(10); p = c.malloc(20); c.malloc(4)
    c.free(p)
    s = c.stats()
    assert s["free"] + s["used"] == 64
    assert s["used"] == 14
    assert s["largest_free"] == 30
    print("Part 2 OK")

_t2()

## Part 3 — Make it thread-safe (concurrent malloc/free)

Wrap your Part 2 allocator as `ThreadSafeAllocator` so concurrent callers can't corrupt the free
list. Then implement `parallel_malloc(allocator, sizes, num_workers)`: a pool of worker threads
pulls requests from a **bounded `queue.Queue`** and shuts down on a sentinel.

**Say this out loud:**
- The invariant the lock protects: **no two live allocations overlap.** A data race on the free
  list would hand the same bytes to two callers.
- One coarse lock around malloc/free is correct and usually the right first answer; mention
  per-arena locks / lock-free free lists as the scaling story.
- Bounded queue = backpressure; one sentinel per worker = clean shutdown (no busy-wait, no poison races).

**Things to discuss:** lock granularity, why the GIL doesn't save you (frees interleave at bytecode
boundaries), how you'd test for races (disjointness of the returned intervals).

In [ ]:
import threading
import queue


class ThreadSafeAllocator:
    """Wrap your Part 2 allocator so concurrent malloc/free are safe."""

    def __init__(self, capacity, policy="first"):
        raise NotImplementedError

    def malloc(self, size, align=1):
        raise NotImplementedError

    def free(self, address):
        raise NotImplementedError

    def stats(self):
        raise NotImplementedError


def parallel_malloc(allocator, sizes, num_workers):
    """Feed `sizes` to `num_workers` threads via a bounded queue (sentinel shutdown).
    Return a list of (index, size, address); address is None if it didn't fit."""
    raise NotImplementedError

In [ ]:
# --- Part 3 tests ---
def _t3():
    # No two concurrent allocations may overlap, even under contention.
    for _ in range(5):
        ts = ThreadSafeAllocator(1000)
        res = parallel_malloc(ts, [8] * 100, num_workers=8)   # 800 bytes, fits in 1000
        addrs = [(addr, size) for _, size, addr in res if addr is not None]
        assert len(addrs) == 100                              # all succeed
        iv = sorted((a, a + s) for a, s in addrs)
        for (lo1, hi1), (lo2, hi2) in zip(iv, iv[1:]):
            assert hi1 <= lo2, "overlapping allocations"
        for a, s in addrs:
            assert 0 <= a and a + s <= 1000

    # Over-subscription: only capacity-worth is handed out, still no overlap.
    ts = ThreadSafeAllocator(80)
    res = parallel_malloc(ts, [8] * 50, num_workers=8)
    ok = [(addr, size) for _, size, addr in res if addr is not None]
    assert len(ok) == 10                                      # 80 / 8
    iv = sorted((a, a + s) for a, s in ok)
    for (l1, h1), (l2, h2) in zip(iv, iv[1:]):
        assert h1 <= l2
    print("Part 3 OK")

_t3()

## Part 4 (stretch) — Hostile, malformed input (streaming)

Real allocators are fed garbage. Add `apply(op)` to a `RobustAllocator` that processes a stream of
operations and **never corrupts internal state**, whatever the input:
- `("malloc", size[, align])` and `("free", addr)`.
- Reject with `("err", reason)`: `bad_size` (≤0 / non-int), `too_big` (> capacity), `bad_align`
  (not a power of two), `oom` (no space), `invalid_free` (double-free, unknown address, or an
  **interior pointer** that isn't a base), `unknown_op`.
- Success returns `("ok", addr)`. Track `self.accepted` / `self.rejected`.

**Say this out loud:** validate *before* mutating; an interior pointer (an address inside a block but
not its base) must be rejected; a double-free is just a free of an address that's no longer allocated.

**Things to discuss:** detecting interior pointers in O(1) (base-address set) vs. O(log n) (interval
tree), and the security angle (use-after-free / double-free exploits).

In [ ]:
class RobustAllocator(Allocator):
    """Add apply(op) that tolerates malformed/hostile requests without corrupting state.

    op is ("malloc", size[, align]) or ("free", address).
    Return ("ok", address) or ("err", reason). Track self.accepted / self.rejected.
    """

    def __init__(self, capacity, policy="first"):
        raise NotImplementedError

    def apply(self, op):
        raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
def _t4():
    def invariants(a):
        cap = a.capacity
        prev_end = -1
        for off, sz in a.free_list:                       # free blocks: sorted, in-range, coalesced
            assert sz > 0 and off >= 0 and off + sz <= cap
            assert off > prev_end, "free blocks not coalesced / overlapping"
            prev_end = off + sz
        spans = [(o, o + s) for o, s in a.free_list] + \
                [(o, o + s) for o, s in a.allocated.items()]
        spans.sort()
        cur = 0
        for lo, hi in spans:                              # free + allocated tile [0, cap) exactly
            assert lo == cur, "gap/overlap at %d" % lo
            cur = hi
        assert cur == cap

    a = RobustAllocator(100)
    assert a.apply(("malloc", 10)) == ("ok", 0)
    assert a.apply(("malloc", 10)) == ("ok", 10)
    assert a.apply(("free", 0)) == ("ok", 0)
    assert a.apply(("free", 0)) == ("err", "invalid_free")     # double free
    assert a.apply(("free", 15)) == ("err", "invalid_free")    # interior pointer (base is 10)
    assert a.apply(("malloc", 0)) == ("err", "bad_size")
    assert a.apply(("malloc", -3)) == ("err", "bad_size")
    assert a.apply(("malloc", 1000)) == ("err", "too_big")
    assert a.apply(("malloc", 8, 3)) == ("err", "bad_align")   # align not a power of two
    assert a.apply(("nope",)) == ("err", "unknown_op")
    invariants(a)

    # A storm of mixed valid/garbage ops must never corrupt the allocator.
    import random
    rng = random.Random(0)
    b = RobustAllocator(256)
    live = []
    for _ in range(4000):
        roll = rng.random()
        if roll < 0.5:
            r = b.apply(("malloc", rng.randint(1, 40), 1 << rng.randint(0, 3)))
            if r[0] == "ok":
                live.append(r[1])
        elif roll < 0.8 and live:
            addr = live.pop(rng.randrange(len(live)))
            assert b.apply(("free", addr)) == ("ok", addr)
        else:
            bad = rng.choice([("free", -1), ("free", 10 ** 9), ("malloc", 0),
                              ("malloc", -5), ("malloc", 10 ** 9), ("explode",)])
            assert b.apply(bad)[0] == "err"
        invariants(b)
    assert b.accepted > 0 and b.rejected > 0
    print("Part 4 OK")

_t4()

## Part 5 (stretch) — Parallel fragmentation simulation (CPU-bound)

To choose a policy you simulate. For each random seed, replay a long random malloc/free workload and
measure peak utilization and failures. That's CPU-bound and embarrassingly parallel — run it across
processes with `ProcessPoolExecutor`.

`allocator_workers.simulate((seed, capacity, n_ops, max_alloc, free_prob))` lives in a real module
(spawned processes can't pickle notebook-defined code). Implement `run_parallel(param_list)` to fan
the seeds across a process pool and collect the metric dicts.

**Say this out loud:** the work must be picklable (hence the module); each seed is deterministic, so
the parallel result must exactly equal the serial one; use processes (not threads) because it's
CPU-bound and the GIL would serialize threads.

**Things to discuss:** chunking seeds to amortize spawn cost, why threads wouldn't help here, and how
you'd shard one big allocator into N arenas (multiplexing) to scale a real workload.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import allocator_workers   # simulate(params) lives here so it's picklable under spawn


def run_parallel(param_list, max_workers=None):
    """Run allocator_workers.simulate on each params tuple across a process pool.
    Return a list of metric dicts, one per params tuple (same order)."""
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    params = [(seed, 1000, 2000, 64, 0.45) for seed in range(8)]
    serial = [allocator_workers.simulate(p) for p in params]
    parallel = run_parallel(params)
    assert parallel == serial            # picklable workers + per-seed determinism
    for r in serial:
        assert r["alloc_ok"] > 0
        assert r["peak_used"] <= 1000
        assert r["free"] >= r["largest_free"] >= 0
    print("Part 5 OK")

_t5()

## Likely follow-ups (be ready to talk through, maybe code)

- **Segregated / buddy allocator:** size classes for O(1) malloc/free; internal vs. external fragmentation.
- **realloc:** grow in place if the next block is free, else move; amortized cost.
- **Compaction:** when you can move live blocks (needs handles/indirection, not raw pointers).
- **Per-thread arenas (tcmalloc / jemalloc):** remove lock contention; periodic rebalancing.
- **Detecting leaks / double-frees** in production; guard pages and red zones.

In [ ]:
# Reference solutions (collapsed at the bottom on purpose — peek only after trying).
import threading
import queue
from concurrent.futures import ProcessPoolExecutor
import allocator_workers


class RefAllocator:
    def __init__(self, capacity, policy="first"):
        self.capacity = capacity
        self.policy = policy
        self.free_list = [(0, capacity)] if capacity > 0 else []
        self.allocated = {}

    def _find(self, size, align):
        best = None
        for i, (off, sz) in enumerate(self.free_list):
            astart = (off + align - 1) // align * align
            pad = astart - off
            if pad + size <= sz:
                if self.policy == "first":
                    return (i, astart, pad)
                waste = sz - (pad + size)
                if best is None or waste < best[0]:
                    best = (waste, i, astart, pad)
        if best is None:
            return None
        _, i, astart, pad = best
        return (i, astart, pad)

    def malloc(self, size, align=1):
        if not isinstance(size, int) or size <= 0 or size > self.capacity:
            return None
        if not isinstance(align, int) or align < 1 or (align & (align - 1)):
            return None
        found = self._find(size, align)
        if found is None:
            return None
        i, astart, pad = found
        off, sz = self.free_list[i]
        rest = []
        if pad:
            rest.append((off, pad))
        tail_off, tail_sz = astart + size, (off + sz) - (astart + size)
        if tail_sz:
            rest.append((tail_off, tail_sz))
        self.free_list[i:i + 1] = rest
        self.allocated[astart] = size
        return astart

    def free(self, address):
        if address not in self.allocated:
            raise KeyError(address)
        size = self.allocated.pop(address)
        lo, hi = 0, len(self.free_list)
        while lo < hi:
            mid = (lo + hi) // 2
            if self.free_list[mid][0] < address:
                lo = mid + 1
            else:
                hi = mid
        self.free_list.insert(lo, (address, size))
        i = lo
        while i + 1 < len(self.free_list) and \
                self.free_list[i][0] + self.free_list[i][1] == self.free_list[i + 1][0]:
            o, s = self.free_list[i]; _, s2 = self.free_list[i + 1]
            self.free_list[i:i + 2] = [(o, s + s2)]
        while i > 0 and \
                self.free_list[i - 1][0] + self.free_list[i - 1][1] == self.free_list[i][0]:
            o, s = self.free_list[i - 1]; _, s2 = self.free_list[i]
            self.free_list[i - 1:i + 1] = [(o, s + s2)]; i -= 1

    def stats(self):
        sizes = [s for _, s in self.free_list]
        free = sum(sizes)
        return {"capacity": self.capacity, "free": free, "used": self.capacity - free,
                "largest_free": max(sizes) if sizes else 0, "num_free_blocks": len(sizes)}


class RefThreadSafeAllocator:
    def __init__(self, capacity, policy="first"):
        self._a = RefAllocator(capacity, policy)
        self._lock = threading.Lock()

    def malloc(self, size, align=1):
        with self._lock:
            return self._a.malloc(size, align)

    def free(self, address):
        with self._lock:
            return self._a.free(address)

    def stats(self):
        with self._lock:
            return self._a.stats()


def ref_parallel_malloc(allocator, sizes, num_workers):
    q = queue.Queue(maxsize=64)
    results, rlock, SENT = [], threading.Lock(), object()

    def worker():
        while True:
            item = q.get()
            if item is SENT:
                q.task_done(); break
            idx, size = item
            addr = allocator.malloc(size)
            with rlock:
                results.append((idx, size, addr))
            q.task_done()

    ts = [threading.Thread(target=worker) for _ in range(num_workers)]
    for t in ts:
        t.start()
    for i, s in enumerate(sizes):
        q.put((i, s))
    for _ in ts:
        q.put(SENT)
    for t in ts:
        t.join()
    return results


class RefRobustAllocator(RefAllocator):
    def __init__(self, capacity, policy="first"):
        super().__init__(capacity, policy)
        self.accepted = self.rejected = 0

    def apply(self, op):
        res = self._apply(op)
        if res[0] == "ok":
            self.accepted += 1
        else:
            self.rejected += 1
        return res

    def _apply(self, op):
        if not op:
            return ("err", "unknown_op")
        if op[0] == "malloc":
            size = op[1] if len(op) > 1 else None
            align = op[2] if len(op) > 2 else 1
            if not isinstance(size, int) or size <= 0:
                return ("err", "bad_size")
            if size > self.capacity:
                return ("err", "too_big")
            if not isinstance(align, int) or align < 1 or (align & (align - 1)):
                return ("err", "bad_align")
            addr = self.malloc(size, align)
            return ("ok", addr) if addr is not None else ("err", "oom")
        if op[0] == "free":
            addr = op[1] if len(op) > 1 else None
            if addr not in self.allocated:
                return ("err", "invalid_free")
            self.free(addr)
            return ("ok", addr)
        return ("err", "unknown_op")


def ref_run_parallel(param_list, max_workers=None):
    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        return list(ex.map(allocator_workers.simulate, param_list))


# --- sanity checks across all five parts ---
_a = RefAllocator(100)
assert [_a.malloc(10), _a.malloc(20), _a.malloc(70), _a.malloc(1)] == [0, 10, 30, None]
_a.free(0); assert _a.malloc(8) == 0
_b = RefAllocator(100, "best")
for _s in (40, 10, 20, 10):
    _b.malloc(_s)
_b.free(0); _b.free(50); assert _b.malloc(20) == 50
assert RefAllocator(100).malloc(8, align=8) == 0
_ts = RefThreadSafeAllocator(1000)
_r = [a for _, _, a in ref_parallel_malloc(_ts, [8] * 100, 8) if a is not None]
assert len(_r) == 100 and len(set(_r)) == 100
_ra = RefRobustAllocator(50)
assert _ra.apply(("malloc", 10)) == ("ok", 0)
assert _ra.apply(("free", 0)) == ("ok", 0)
assert _ra.apply(("free", 0)) == ("err", "invalid_free")
assert _ra.apply(("malloc", 0)) == ("err", "bad_size")
_params = [(s, 1000, 1500, 64, 0.45) for s in range(4)]
assert ref_run_parallel(_params) == [allocator_workers.simulate(p) for p in _params]
print("reference solutions OK")